In [3]:
# ============================================================
# V5 — CONTEXTUAL BANDIT LLM ROUTER
# ============================================================
#
# V4:
#   Learn model performance
#        ↓
#   Fixed routing policy
#
# V5:
#   V4 learned performance
#        ↓
#   Contextual epsilon-greedy bandit
#        ↓
#   EXPLORE ↔ EXPLOIT
#        ↓
#   Observe selected-model feedback
#        ↓
#   Update reward estimate
#
# AWS:
#   SageMaker + S3 + CloudWatch
#
# Context:
#   query complexity
#
# Arms:
#   small_model
#   medium_model
#   large_model
#
# ============================================================

import os
import json
import boto3
import numpy as np
import pandas as pd

from botocore.exceptions import ClientError


# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_ROOT = (
    "/home/sagemaker-user/"
    "adaptive-llm-routing"
)

V4_DIR = os.path.join(
    PROJECT_ROOT,
    "notebooks",
    "data",
    "evaluation",
    "v4"
)

V5_DIR = os.path.join(
    PROJECT_ROOT,
    "notebooks",
    "data",
    "evaluation",
    "v5"
)

os.makedirs(
    V5_DIR,
    exist_ok=True
)


RANDOM_SEED = 42

TRAIN_FRACTION = 0.70

INITIAL_EPSILON = 0.20

MIN_EPSILON = 0.05

EPSILON_DECAY = 0.985

PRIOR_COUNT = 10

COST_PENALTY = 0.10

LATENCY_PENALTY = 0.10

QUALITY_VIOLATION_PENALTY = 1.00

SUCCESS_THRESHOLD = 0.95


MODELS = [
    "small_model",
    "medium_model",
    "large_model"
]


QUALITY_REQUIREMENT = {

    "simple": 0.80,

    "medium": 0.90,

    "complex": 0.95
}


print("=" * 100)
print("V5 — CONTEXTUAL BANDIT LLM ROUTER")
print("=" * 100)


# ============================================================
# 2. AWS SESSION
# ============================================================

session = boto3.Session()

region = session.region_name

if region is None:

    raise RuntimeError(
        "Could not determine AWS region."
    )


sts = session.client(
    "sts"
)

account_id = (
    sts.get_caller_identity()
    ["Account"]
)


bucket_name = (
    f"adaptive-llm-router-"
    f"{account_id}-"
    f"{region}"
).lower()


s3 = session.client(
    "s3",
    region_name=region
)

cloudwatch = session.client(
    "cloudwatch",
    region_name=region
)


S3_V4_PREFIX = (
    "adaptive-llm-router/v4"
)

S3_V5_PREFIX = (
    "adaptive-llm-router/v5"
)


print("\nAWS region:")
print(region)

print("\nS3 bucket:")
print(bucket_name)


# ============================================================
# 3. DOWNLOAD V4 FEEDBACK FROM S3
# ============================================================
#
# V5 explicitly uses the V4 data stored in S3.
# ============================================================

v4_s3_key = (
    S3_V4_PREFIX
    +
    "/routing_feedback.csv"
)

v5_local_feedback = os.path.join(
    V5_DIR,
    "v4_feedback_source.csv"
)


try:

    s3.download_file(

        bucket_name,

        v4_s3_key,

        v5_local_feedback
    )

    print(
        "\nDownloaded V4 feedback from S3."
    )

    print(
        f"s3://{bucket_name}/{v4_s3_key}"
    )


except ClientError as e:

    print(
        "\nS3 download failed."
    )

    print(
        "Using local V4 feedback instead."
    )

    v5_local_feedback = os.path.join(
        V4_DIR,
        "routing_feedback.csv"
    )


if not os.path.exists(
    v5_local_feedback
):

    raise FileNotFoundError(
        "Could not find V4 feedback data."
    )


feedback_df = pd.read_csv(
    v5_local_feedback
)


print(
    "\nV4 feedback records:",
    len(feedback_df)
)


# ============================================================
# 4. VALIDATE FEEDBACK DATA
# ============================================================

required_columns = {

    "query_id",
    "query",
    "complexity",
    "model",
    "quality",
    "latency_ms",
    "cost",
    "success"
}


missing_columns = (
    required_columns
    -
    set(
        feedback_df.columns
    )
)


if missing_columns:

    raise ValueError(
        f"Missing columns: {missing_columns}"
    )


print(
    "\nFeedback columns validated."
)


# ============================================================
# 5. RECREATE V4 TRAIN / HOLDOUT SPLIT
# ============================================================
#
# Same seed and same methodology as V4.
# This keeps V4 and V5 directly comparable.
# ============================================================

query_df = (
    feedback_df[
        [
            "query_id",
            "query",
            "complexity"
        ]
    ]
    .drop_duplicates(
        subset=[
            "query_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


rng_split = np.random.default_rng(
    RANDOM_SEED
)


train_query_ids = []

holdout_query_ids = []


for complexity in [
    "simple",
    "medium",
    "complex"
]:

    ids = (
        query_df.loc[
            query_df[
                "complexity"
            ]
            ==
            complexity,

            "query_id"
        ]
        .to_numpy(
            copy=True
        )
    )


    rng_split.shuffle(
        ids
    )


    split_index = int(
        len(ids)
        *
        TRAIN_FRACTION
    )


    train_query_ids.extend(
        ids[
            :split_index
        ]
    )

    holdout_query_ids.extend(
        ids[
            split_index:
        ]
    )


train_query_ids = set(
    train_query_ids
)

holdout_query_ids = set(
    holdout_query_ids
)


train_feedback = feedback_df[
    feedback_df[
        "query_id"
    ].isin(
        train_query_ids
    )
].copy()


holdout_feedback = feedback_df[
    feedback_df[
        "query_id"
    ].isin(
        holdout_query_ids
    )
].copy()


holdout_queries = (
    holdout_feedback[
        [
            "query_id",
            "query",
            "complexity"
        ]
    ]
    .drop_duplicates(
        subset=[
            "query_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


print("\n")
print("=" * 100)
print("V5 TRAIN / HOLDOUT")
print("=" * 100)

print(
    "Training queries:",
    len(train_query_ids)
)

print(
    "Holdout queries:",
    len(holdout_query_ids)
)

print(
    "Holdout requests:",
    len(holdout_queries)
)


# ============================================================
# 6. LEARN V4 PERFORMANCE PROFILE
# ============================================================
#
# V5 starts with knowledge learned by V4.
# ============================================================

performance_df = (

    train_feedback

    .groupby(
        [
            "complexity",
            "model"
        ]
    )

    .agg(

        queries=(
            "query_id",
            "nunique"
        ),

        avg_quality=(
            "quality",
            "mean"
        ),

        avg_latency_ms=(
            "latency_ms",
            "mean"
        ),

        avg_cost=(
            "cost",
            "mean"
        ),

        success_rate=(
            "success",
            "mean"
        )
    )

    .reset_index()
)


# ============================================================
# 7. NORMALIZATION RANGES
# ============================================================

cost_min = (
    performance_df[
        "avg_cost"
    ].min()
)

cost_max = (
    performance_df[
        "avg_cost"
    ].max()
)

latency_min = (
    performance_df[
        "avg_latency_ms"
    ].min()
)

latency_max = (
    performance_df[
        "avg_latency_ms"
    ].max()
)


def normalize(
    value,
    minimum,
    maximum
):

    if maximum == minimum:

        return 0.0

    return (
        value - minimum
    ) / (
        maximum - minimum
    )


# ============================================================
# 8. REWARD FUNCTION
# ============================================================
#
# Reward balances:
#
#   quality
#   cost
#   latency
#   minimum quality requirement
#
# A model that is fast/cheap but fails the required quality
# receives a quality-violation penalty.
# ============================================================

def calculate_reward(
    complexity,
    quality,
    latency_ms,
    cost
):

    required_quality = (
        QUALITY_REQUIREMENT[
            complexity
        ]
    )


    normalized_cost = normalize(
        cost,
        cost_min,
        cost_max
    )


    normalized_latency = normalize(
        latency_ms,
        latency_min,
        latency_max
    )


    quality_violation = max(

        0.0,

        required_quality
        -
        quality
    )


    reward = (

        quality

        -
        COST_PENALTY
        *
        normalized_cost

        -
        LATENCY_PENALTY
        *
        normalized_latency

        -
        QUALITY_VIOLATION_PENALTY
        *
        quality_violation
    )


    return float(
        np.clip(
            reward,
            -1.0,
            1.0
        )
    )


# ============================================================
# 9. BUILD V4-INITIALIZED BANDIT STATE
# ============================================================
#
# Context = complexity
#
# Each context has one estimate for each model.
#
# Example:
#
# simple:
#   small   → estimated reward
#   medium  → estimated reward
#   large   → estimated reward
#
# V5 starts with V4's observed estimates.
# ============================================================

bandit_state = {}


for complexity in [
    "simple",
    "medium",
    "complex"
]:

    bandit_state[
        complexity
    ] = {}


    for model_name in MODELS:

        row = performance_df[
            (
                performance_df[
                    "complexity"
                ]
                ==
                complexity
            )
            &
            (
                performance_df[
                    "model"
                ]
                ==
                model_name
            )
        ]


        if len(row) != 1:

            raise RuntimeError(
                f"Missing V4 profile for "
                f"{complexity}/{model_name}"
            )


        row = row.iloc[0]


        initial_reward = (
            calculate_reward(

                complexity,

                row[
                    "avg_quality"
                ],

                row[
                    "avg_latency_ms"
                ],

                row[
                    "avg_cost"
                ]
            )
        )


        bandit_state[
            complexity
        ][model_name] = {

            "estimated_reward":
                initial_reward,

            "count":
                PRIOR_COUNT,

            "v4_quality":
                float(
                    row[
                        "avg_quality"
                    ]
                ),

            "v4_latency_ms":
                float(
                    row[
                        "avg_latency_ms"
                    ]
                ),

            "v4_cost":
                float(
                    row[
                        "avg_cost"
                    ]
                )
        }


# ============================================================
# 10. EPSILON-GREEDY CONTEXTUAL BANDIT
# ============================================================

class ContextualEpsilonGreedy:

    def __init__(
        self,
        state,
        initial_epsilon=0.20,
        min_epsilon=0.05,
        decay=0.985,
        seed=42
    ):

        self.state = state

        self.epsilon = (
            initial_epsilon
        )

        self.initial_epsilon = (
            initial_epsilon
        )

        self.min_epsilon = (
            min_epsilon
        )

        self.decay = decay

        self.rng = (
            np.random.default_rng(
                seed
            )
        )


    def select(
        self,
        context
    ):

        values = {

            model:
                self.state[
                    context
                ][
                    model
                ][
                    "estimated_reward"
                ]

            for model in MODELS
        }


        # ----------------------------------------------------
        # Explore
        # ----------------------------------------------------

        if (
            self.rng.random()
            <
            self.epsilon
        ):

            selected = (
                self.rng.choice(
                    MODELS
                )
            )

            exploration = True


        # ----------------------------------------------------
        # Exploit
        # ----------------------------------------------------

        else:

            max_value = max(
                values.values()
            )


            best_models = [

                model

                for model, value
                in values.items()

                if value
                ==
                max_value
            ]


            selected = (
                self.rng.choice(
                    best_models
                )
            )

            exploration = False


        return (
            selected,
            exploration,
            values
        )


    def update(
        self,
        context,
        model,
        reward
    ):

        state = (
            self.state[
                context
            ][
                model
            ]
        )


        count = (
            state[
                "count"
            ]
        )


        old_value = (
            state[
                "estimated_reward"
            ]
        )


        new_count = (
            count
            +
            1
        )


        new_value = (

            old_value
            +
            (
                reward
                -
                old_value
            )
            /
            new_count
        )


        state[
            "estimated_reward"
        ] = new_value


        state[
            "count"
        ] = new_count


    def decay_epsilon(
        self
    ):

        self.epsilon = max(

            self.min_epsilon,

            self.epsilon
            *
            self.decay
        )


# ============================================================
# 11. INITIALIZE BANDIT
# ============================================================

bandit = ContextualEpsilonGreedy(

    state=bandit_state,

    initial_epsilon=INITIAL_EPSILON,

    min_epsilon=MIN_EPSILON,

    decay=EPSILON_DECAY,

    seed=RANDOM_SEED
)


# ============================================================
# 12. RUN V5 ON HOLDOUT
# ============================================================
#
# IMPORTANT:
#
# The bandit only receives the feedback from the model it
# selected.
#
# It does NOT update itself using the unselected models'
# outcomes.
# ============================================================

v5_results = []

cumulative_reward = 0.0

total_reward_history = []

exploration_count = 0

exploitation_count = 0


for step, (_, row) in enumerate(
    holdout_queries.iterrows(),
    start=1
):

    query_id = int(
        row[
            "query_id"
        ]
    )

    query = row[
        "query"
    ]

    complexity = row[
        "complexity"
    ]


    # --------------------------------------------------------
    # Bandit decision
    # --------------------------------------------------------

    selected_model, exploration, estimates = (
        bandit.select(
            complexity
        )
    )


    if exploration:

        exploration_count += 1

    else:

        exploitation_count += 1


    # --------------------------------------------------------
    # Reveal ONLY selected-model outcome
    # --------------------------------------------------------

    actual = holdout_feedback[

        (
            holdout_feedback[
                "query_id"
            ]
            ==
            query_id
        )

        &

        (
            holdout_feedback[
                "model"
            ]
            ==
            selected_model
        )
    ]


    if len(actual) != 1:

        raise RuntimeError(
            "Expected exactly one "
            "selected-model outcome."
        )


    actual = actual.iloc[0]


    quality = float(
        actual[
            "quality"
        ]
    )

    latency = float(
        actual[
            "latency_ms"
        ]
    )

    cost = float(
        actual[
            "cost"
        ]
    )

    success = int(
        actual[
            "success"
        ]
    )


    # --------------------------------------------------------
    # Calculate reward
    # --------------------------------------------------------

    reward = calculate_reward(

        complexity,

        quality,

        latency,

        cost
    )


    # --------------------------------------------------------
    # Update bandit
    # --------------------------------------------------------

    bandit.update(

        complexity,

        selected_model,

        reward
    )


    # --------------------------------------------------------
    # Decay exploration
    # --------------------------------------------------------

    bandit.decay_epsilon()


    cumulative_reward += reward


    total_reward_history.append(
        cumulative_reward
    )


    v5_results.append({

        "step":
            step,

        "query_id":
            query_id,

        "query":
            query,

        "complexity":
            complexity,

        "selected_model":
            selected_model,

        "exploration":
            exploration,

        "epsilon":
            bandit.epsilon,

        "quality":
            quality,

        "latency_ms":
            latency,

        "cost":
            cost,

        "success":
            success,

        "reward":
            reward,

        "cumulative_reward":
            cumulative_reward
    })


v5_results_df = pd.DataFrame(
    v5_results
)


# ============================================================
# 13. BUILD V4 FIXED POLICY FOR SAME HOLDOUT
# ============================================================
#
# V4 uses its learned training performance but DOES NOT
# update during the holdout.
# ============================================================

def v4_fixed_route(
    complexity
):

    required_quality = (
        QUALITY_REQUIREMENT[
            complexity
        ]
    )


    candidates = []


    for model_name in MODELS:

        row = performance_df[

            (
                performance_df[
                    "complexity"
                ]
                ==
                complexity
            )

            &

            (
                performance_df[
                    "model"
                ]
                ==
                model_name
            )
        ]


        row = row.iloc[0]


        if (

            row[
                "avg_quality"
            ]
            >=
            required_quality

            and

            row[
                "success_rate"
            ]
            >=
            SUCCESS_THRESHOLD
        ):

            candidates.append(
                model_name
            )


    if not candidates:

        return max(

            MODELS,

            key=lambda model:

            float(
                performance_df[
                    (
                        performance_df[
                            "complexity"
                        ]
                        ==
                        complexity
                    )

                    &

                    (
                        performance_df[
                            "model"
                        ]
                        ==
                        model
                    )
                ]
                [
                    "avg_quality"
                ]
                .iloc[0]
            )
        )


    candidate_rows = performance_df[
        performance_df[
            "model"
        ].isin(
            candidates
        )

        &

        (
            performance_df[
                "complexity"
            ]
            ==
            complexity
        )
    ].copy()


    candidate_rows[
        "cost_norm"
    ] = (

        candidate_rows[
            "avg_cost"
        ]
        -
        cost_min

    ) / (

        cost_max
        -
        cost_min
    )


    candidate_rows[
        "latency_norm"
    ] = (

        candidate_rows[
            "avg_latency_ms"
        ]
        -
        latency_min

    ) / (

        latency_max
        -
        latency_min
    )


    candidate_rows[
        "operational_score"
    ] = (

        COST_PENALTY
        *
        candidate_rows[
            "cost_norm"
        ]

        +

        LATENCY_PENALTY
        *
        candidate_rows[
            "latency_norm"
        ]
    )


    return (
        candidate_rows
        .sort_values(
            "operational_score"
        )
        .iloc[0]
        [
            "model"
        ]
    )


# ============================================================
# 14. EVALUATE V4 ON HOLDOUT
# ============================================================

v4_results = []


for _, row in (
    holdout_queries.iterrows()
):

    query_id = int(
        row[
            "query_id"
        ]
    )

    complexity = row[
        "complexity"
    ]


    selected_model = (
        v4_fixed_route(
            complexity
        )
    )


    actual = holdout_feedback[

        (
            holdout_feedback[
                "query_id"
            ]
            ==
            query_id
        )

        &

        (
            holdout_feedback[
                "model"
            ]
            ==
            selected_model
        )
    ]


    actual = actual.iloc[0]


    v4_reward = calculate_reward(

        complexity,

        float(
            actual[
                "quality"
            ]
        ),

        float(
            actual[
                "latency_ms"
            ]
        ),

        float(
            actual[
                "cost"
            ]
        )
    )


    v4_results.append({

        "query_id":
            query_id,

        "complexity":
            complexity,

        "selected_model":
            selected_model,

        "quality":
            actual[
                "quality"
            ],

        "latency_ms":
            actual[
                "latency_ms"
            ],

        "cost":
            actual[
                "cost"
            ],

        "success":
            actual[
                "success"
            ],

        "reward":
            v4_reward
    })


v4_results_df = pd.DataFrame(
    v4_results
)


# ============================================================
# 15. V4 METRICS
# ============================================================

v4_avg_quality = (
    v4_results_df[
        "quality"
    ].mean()
)

v4_avg_latency = (
    v4_results_df[
        "latency_ms"
    ].mean()
)

v4_avg_cost = (
    v4_results_df[
        "cost"
    ].mean()
)

v4_success_rate = (
    v4_results_df[
        "success"
    ].mean()
)

v4_avg_reward = (
    v4_results_df[
        "reward"
    ].mean()
)


# ============================================================
# 16. V5 METRICS
# ============================================================

v5_avg_quality = (
    v5_results_df[
        "quality"
    ].mean()
)

v5_avg_latency = (
    v5_results_df[
        "latency_ms"
    ].mean()
)

v5_avg_cost = (
    v5_results_df[
        "cost"
    ].mean()
)

v5_success_rate = (
    v5_results_df[
        "success"
    ].mean()
)

v5_avg_reward = (
    v5_results_df[
        "reward"
    ].mean()
)

v5_exploration_rate = (
    v5_results_df[
        "exploration"
    ].mean()
)

v5_final_epsilon = (
    v5_results_df[
        "epsilon"
    ].iloc[-1]
)


# ============================================================
# 17. V4 VS V5
# ============================================================

comparison_df = pd.DataFrame({

    "metric": [

        "average_quality",

        "average_latency_ms",

        "average_cost",

        "success_rate",

        "average_reward"
    ],

    "V4": [

        v4_avg_quality,

        v4_avg_latency,

        v4_avg_cost,

        v4_success_rate,

        v4_avg_reward
    ],

    "V5": [

        v5_avg_quality,

        v5_avg_latency,

        v5_avg_cost,

        v5_success_rate,

        v5_avg_reward
    ]
})


comparison_df[
    "delta_V5_minus_V4"
] = (

    comparison_df[
        "V5"
    ]

    -

    comparison_df[
        "V4"
    ]
).round(4)


comparison_df[
    "V4"
] = (
    comparison_df[
        "V4"
    ].round(4)
)

comparison_df[
    "V5"
] = (
    comparison_df[
        "V5"
    ].round(4)
)


# ============================================================
# 18. MODEL USAGE
# ============================================================

v4_usage = (
    v4_results_df[
        "selected_model"
    ]
    .value_counts()
    .to_frame(
        "requests"
    )
)

v4_usage[
    "percentage"
] = (

    v4_usage[
        "requests"
    ]

    /

    len(
        v4_results_df
    )

    *

    100
).round(2)


v5_usage = (
    v5_results_df[
        "selected_model"
    ]
    .value_counts()
    .to_frame(
        "requests"
    )
)

v5_usage[
    "percentage"
] = (

    v5_usage[
        "requests"
    ]

    /

    len(
        v5_results_df
    )

    *

    100
).round(2)


# ============================================================
# 19. FINAL BANDIT STATE
# ============================================================

bandit_state_rows = []


for complexity in [
    "simple",
    "medium",
    "complex"
]:

    for model_name in MODELS:

        state = (
            bandit.state[
                complexity
            ][
                model_name
            ]
        )


        bandit_state_rows.append({

            "complexity":
                complexity,

            "model":
                model_name,

            "estimated_reward":
                state[
                    "estimated_reward"
                ],

            "count":
                state[
                    "count"
                ]
        })


bandit_state_df = pd.DataFrame(
    bandit_state_rows
)


# ============================================================
# 20. PRINT RESULTS
# ============================================================

print("\n")
print("=" * 100)
print("V5 BANDIT PERFORMANCE")
print("=" * 100)

print(
    f"Average quality : "
    f"{v5_avg_quality:.4f}"
)

print(
    f"Average latency : "
    f"{v5_avg_latency:.2f} ms"
)

print(
    f"Average cost    : "
    f"{v5_avg_cost:.4f}"
)

print(
    f"Success rate    : "
    f"{v5_success_rate:.4f}"
)

print(
    f"Average reward  : "
    f"{v5_avg_reward:.4f}"
)

print(
    f"Exploration rate: "
    f"{v5_exploration_rate:.4f}"
)

print(
    f"Final epsilon   : "
    f"{v5_final_epsilon:.4f}"
)


print("\n")
print("=" * 100)
print("V4 VS V5")
print("=" * 100)

print(
    comparison_df.to_string(
        index=False
    )
)


print("\n")
print("=" * 100)
print("V4 MODEL USAGE")
print("=" * 100)

print(
    v4_usage
)


print("\n")
print("=" * 100)
print("V5 MODEL USAGE")
print("=" * 100)

print(
    v5_usage
)


print("\n")
print("=" * 100)
print("FINAL BANDIT STATE")
print("=" * 100)

print(
    bandit_state_df.to_string(
        index=False
    )
)


# ============================================================
# 21. SAVE V5 ARTIFACTS
# ============================================================

v5_results_path = os.path.join(
    V5_DIR,
    "v5_bandit_results.csv"
)

v5_results_df.to_csv(
    v5_results_path,
    index=False
)


v4_results_path = os.path.join(
    V5_DIR,
    "v4_holdout_baseline.csv"
)

v4_results_df.to_csv(
    v4_results_path,
    index=False
)


comparison_path = os.path.join(
    V5_DIR,
    "v4_vs_v5_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)


bandit_state_path = os.path.join(
    V5_DIR,
    "v5_bandit_state.csv"
)

bandit_state_df.to_csv(
    bandit_state_path,
    index=False
)


usage_v4_path = os.path.join(
    V5_DIR,
    "v4_model_usage.csv"
)

v4_usage.to_csv(
    usage_v4_path
)


usage_v5_path = os.path.join(
    V5_DIR,
    "v5_model_usage.csv"
)

v5_usage.to_csv(
    usage_v5_path
)


config = {

    "version":
        "V5",

    "algorithm":
        "epsilon_greedy_contextual_bandit",

    "context":
        "query_complexity",

    "models":
        MODELS,

    "initial_epsilon":
        INITIAL_EPSILON,

    "minimum_epsilon":
        MIN_EPSILON,

    "epsilon_decay":
        EPSILON_DECAY,

    "prior_count":
        PRIOR_COUNT,

    "cost_penalty":
        COST_PENALTY,

    "latency_penalty":
        LATENCY_PENALTY,

    "quality_violation_penalty":
        QUALITY_VIOLATION_PENALTY,

    "success_threshold":
        SUCCESS_THRESHOLD,

    "evaluation_queries":
        len(
            holdout_queries
        ),

    "aws_region":
        region,

    "s3_bucket":
        bucket_name,

    "s3_prefix":
        S3_V5_PREFIX
}


config_path = os.path.join(
    V5_DIR,
    "v5_config.json"
)


with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )


# ============================================================
# 22. UPLOAD V5 ARTIFACTS TO S3
# ============================================================

v5_artifacts = [

    "v5_bandit_results.csv",

    "v4_holdout_baseline.csv",

    "v4_vs_v5_comparison.csv",

    "v5_bandit_state.csv",

    "v4_model_usage.csv",

    "v5_model_usage.csv",

    "v5_config.json"
]


print("\n")
print("=" * 100)
print("UPLOADING V5 ARTIFACTS TO S3")
print("=" * 100)


for filename in v5_artifacts:

    local_path = os.path.join(
        V5_DIR,
        filename
    )


    s3_key = (
        S3_V5_PREFIX
        +
        "/"
        +
        filename
    )


    try:

        s3.upload_file(

            local_path,

            bucket_name,

            s3_key
        )


        print(
            f"Uploaded: "
            f"s3://{bucket_name}/"
            f"{s3_key}"
        )


    except ClientError as e:

        print(
            f"Upload failed for "
            f"{filename}: {e}"
        )


# ============================================================
# 23. CLOUDWATCH METRICS
# ============================================================

print("\n")
print("=" * 100)
print("CLOUDWATCH METRICS")
print("=" * 100)


metric_data = [

    {
        "MetricName":
            "AverageQuality",

        "Value":
            float(
                v5_avg_quality
            ),

        "Unit":
            "None",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V5"
            }
        ]
    },

    {
        "MetricName":
            "AverageReward",

        "Value":
            float(
                v5_avg_reward
            ),

        "Unit":
            "None",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V5"
            }
        ]
    },

    {
        "MetricName":
            "AverageLatencyMs",

        "Value":
            float(
                v5_avg_latency
            ),

        "Unit":
            "Milliseconds",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V5"
            }
        ]
    },

    {
        "MetricName":
            "AverageCost",

        "Value":
            float(
                v5_avg_cost
            ),

        "Unit":
            "None",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V5"
            }
        ]
    },

    {
        "MetricName":
            "SuccessRate",

        "Value":
            float(
                v5_success_rate
            ),

        "Unit":
            "None",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V5"
            }
        ]
    },

    {
        "MetricName":
            "ExplorationRate",

        "Value":
            float(
                v5_exploration_rate
            ),

        "Unit":
            "None",

        "Dimensions": [

            {
                "Name":
                    "Version",

                "Value":
                    "V5"
            }
        ]
    }
]


try:

    cloudwatch.put_metric_data(

        Namespace=
            "AdaptiveLLMRouter/V5",

        MetricData=
            metric_data
    )


    print(
        "CloudWatch metrics published successfully."
    )


except ClientError as e:

    print(
        "CloudWatch upload failed."
    )

    print(
        "Check the SageMaker role for "
        "cloudwatch:PutMetricData."
    )

    print(
        e
    )


# ============================================================
# 24. FINAL STATUS
# ============================================================

print("\n")
print("=" * 100)
print("V5 COMPLETE")
print("=" * 100)

print(
    "\nV5 algorithm:"
)

print(
    "Contextual epsilon-greedy bandit"
)

print(
    "Context:",
    "query complexity"
)

print(
    "Exploration:",
    f"{v5_exploration_rate:.2%}"
)

print(
    "Final epsilon:",
    f"{v5_final_epsilon:.4f}"
)


print(
    "\nV4 average reward:",
    round(
        v4_avg_reward,
        4
    )
)

print(
    "V5 average reward:",
    round(
        v5_avg_reward,
        4
    )
)


print(
    "\nS3:"
)

print(
    f"s3://{bucket_name}/{S3_V5_PREFIX}/"
)


print(
    "\nCloudWatch namespace:"
)

print(
    "AdaptiveLLMRouter/V5"
)


print(
    "\nLocal V5 artifacts:"
)

for filename in v5_artifacts:

    print(
        os.path.join(
            V5_DIR,
            filename
        )
    )


print(
    "\nV5 is complete."
)

V5 — CONTEXTUAL BANDIT LLM ROUTER

AWS region:
eu-north-1

S3 bucket:
adaptive-llm-router-753176172779-eu-north-1

Downloaded V4 feedback from S3.
s3://adaptive-llm-router-753176172779-eu-north-1/adaptive-llm-router/v4/routing_feedback.csv

V4 feedback records: 1917

Feedback columns validated.


V5 TRAIN / HOLDOUT
Training queries: 447
Holdout queries: 192
Holdout requests: 192


V5 BANDIT PERFORMANCE
Average quality : 0.9215
Average latency : 652.29 ms
Average cost    : 0.4943
Success rate    : 1.0000
Average reward  : 0.8207
Exploration rate: 0.0677
Final epsilon   : 0.0500


V4 VS V5
            metric       V4       V5  delta_V5_minus_V4
   average_quality   0.9251   0.9215            -0.0036
average_latency_ms 638.8798 652.2870            13.4072
      average_cost   0.4800   0.4943             0.0142
      success_rate   1.0000   1.0000             0.0000
    average_reward   0.8316   0.8207            -0.0109


V4 MODEL USAGE
                requests  percentage
selected_model 

In [2]:
import boto3
import uuid

s3 = boto3.client("s3")
cloudwatch = boto3.client("cloudwatch")

bucket = "adaptive-llm-router-753176172779-eu-north-1"

# Test V5 S3 write
key = f"adaptive-llm-router/v5/test-{uuid.uuid4().hex}.txt"

s3.put_object(
    Bucket=bucket,
    Key=key,
    Body=b"V5 permission test"
)

print("S3 V5 write: SUCCESS")

# Test CloudWatch
cloudwatch.put_metric_data(
    Namespace="AdaptiveLLMRouter/V5",
    MetricData=[
        {
            "MetricName": "PermissionTest",
            "Value": 1,
            "Unit": "Count"
        }
    ]
)

print("CloudWatch metrics: SUCCESS")

S3 V5 write: SUCCESS
CloudWatch metrics: SUCCESS
